In [5]:
import numpy as np

data_path = BASE_DIR / "master_table_station_hour_2022_2024_benchmark_labeled.csv"

df = pd.read_csv(data_path, low_memory=False)

print(df.shape)
print(df["traffic_change_label"].value_counts())

(1395190, 71)
traffic_change_label
normal      722252
decrease    327586
increase    290250
Name: count, dtype: int64


In [6]:
# ==============================
# Step 1: Prepare data for region sensitivity table
# ==============================

work_df = df.copy()

# Use weather_combined_label as weather_condition
work_df["weather_condition"] = work_df["weather_combined_label"]

# Convert relevant columns to numeric
numeric_cols = [
    "volume",
    "expected_volume",
    "traffic_change_pct",
    "temperature_2m_c",
    "rain_mm"
]

for col in numeric_cols:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

# Define significant change based on the new 3-category labels
# New labels: decrease / normal / increase
work_df["has_significant_change_clean"] = work_df["traffic_change_label"].isin(
    ["decrease", "increase"]
)

# Keep rows needed for LGA-level aggregation
work_df = work_df.dropna(
    subset=[
        "lga",
        "weather_condition",
        "station_key",
        "volume",
        "expected_volume",
        "traffic_change_pct"
    ]
)

print(work_df.shape)
work_df[[
    "lga",
    "weather_condition",
    "station_key",
    "volume",
    "expected_volume",
    "traffic_change_pct",
    "traffic_change_label"
]].head()

(1340088, 73)


,lga,weather_condition,station_key,volume,expected_volume,traffic_change_pct,traffic_change_label
0,Sydney,no_rain+very_humid,55306,268.0,988.0,-72.87,decrease
1,Sydney,no_rain+very_humid,55306,152.0,566.0,-73.14,decrease
2,Sydney,no_rain+very_humid,55306,136.0,398.5,-65.87,decrease
3,Sydney,no_rain+very_humid,55306,103.0,281.5,-63.41,decrease
4,Sydney,no_rain+very_humid,55306,124.0,235.0,-47.23,decrease


In [7]:
# ==============================
# Step 2: Generate region sensitivity table
# ==============================

region_sensitivity_table = (
    work_df
    .groupby(["lga", "weather_condition"])
    .agg(
        n_observations=("station_key", "count"),
        n_stations=("station_key", "nunique"),
        avg_expected_volume=("expected_volume", "mean"),
        avg_observed_volume=("volume", "mean"),
        avg_temperature_c=("temperature_2m_c", "mean"),
        avg_rain_mm=("rain_mm", "mean"),
        sensitivity_score=("traffic_change_pct", lambda x: np.mean(np.abs(x))),
        median_abs_change_pct=("traffic_change_pct", lambda x: np.median(np.abs(x))),
        significant_change_rate=("has_significant_change_clean", "mean")
    )
    .reset_index()
)

# Sort by sensitivity score
region_sensitivity_table = region_sensitivity_table.sort_values(
    by="sensitivity_score",
    ascending=False
)

print(region_sensitivity_table.shape)
region_sensitivity_table.head()

(1079, 11)


,lga,weather_condition,n_observations,n_stations,avg_expected_volume,avg_observed_volume,avg_temperature_c,avg_rain_mm,sensitivity_score,median_abs_change_pct,significant_change_rate
799,Strathfield,heavy_rain+very_humid,1,1,225.000000,596.000000,17.5,9.900000,164.89,164.89,1.0
863,Sutherland,no_rain+extreme_heat+strong_wind+overcast,1,1,767.500000,1674.000000,35.0,0.000000,118.11,118.11,1.0
798,Strathfield,heavy_rain+strong_wind+very_humid+overcast,1,1,186.000000,17.000000,19.6,12.200000,90.86,90.86,1.0
675,Parramatta,no_rain+strong_wind+very_humid,3,3,960.833333,1826.666667,19.3,0.033333,85.64,60.32,1.0
97,Blacktown,no_rain+extreme_heat+strong_wind+overcast,1,1,498.000000,75.000000,35.2,0.000000,84.94,84.94,1.0


In [8]:
# ==============================
# Step 3: Save output
# ==============================

output_path = BASE_DIR / "region_sensitivity_table.csv"

region_sensitivity_table.to_csv(output_path, index=False)

print("Saved to:", output_path)
region_sensitivity_table.head()

Saved to: region_sensitivity_table.csv


,lga,weather_condition,n_observations,n_stations,avg_expected_volume,avg_observed_volume,avg_temperature_c,avg_rain_mm,sensitivity_score,median_abs_change_pct,significant_change_rate
799,Strathfield,heavy_rain+very_humid,1,1,225.000000,596.000000,17.5,9.900000,164.89,164.89,1.0
863,Sutherland,no_rain+extreme_heat+strong_wind+overcast,1,1,767.500000,1674.000000,35.0,0.000000,118.11,118.11,1.0
798,Strathfield,heavy_rain+strong_wind+very_humid+overcast,1,1,186.000000,17.000000,19.6,12.200000,90.86,90.86,1.0
675,Parramatta,no_rain+strong_wind+very_humid,3,3,960.833333,1826.666667,19.3,0.033333,85.64,60.32,1.0
97,Blacktown,no_rain+extreme_heat+strong_wind+overcast,1,1,498.000000,75.000000,35.2,0.000000,84.94,84.94,1.0
